# 1.8 - Probability & Statistics for AI Engineers

**Module 1 - Math Prerequisites** | Netsetos GenAI Engineering

This notebook turns four probability ideas into runnable code you can poke at: it builds a categorical PMF the way an LLM samples tokens, simulates log-normal API latency to expose why the mean lies, exercises the four distributions you actually reach for in production (Bernoulli, Poisson, Gaussian, Beta), and hand-rolls the STFT that feeds a speech encoder. Every block is pure NumPy/SciPy - no API keys, no GPU.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

Install the one third-party dependency this lesson leans on. Everything downstream is NumPy (plus SciPy, which ships in most notebook environments); run this once per environment and skip it if NumPy is already present.

In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q numpy


**Explanation**

A one-line environment-prep cell, not lesson logic. It shells out to pip to guarantee NumPy is available before any of the probability code runs.

**How the code works - step by step**
- **`!pip install -q numpy`** - installs NumPy quietly (`-q` suppresses the pip logs); drop the flag if you want to see the full install trace.

**In one line:** make sure NumPy is importable, then move on.

**What you'll see:** (no output - environment prep)

## 1 - A categorical PMF is how an LLM picks its next token

A discrete random variable has countable outcomes and a PMF that sums to 1. The cleanest example in all of GenAI is next-token prediction: softmax turns raw logits into exactly that PMF, and `np.random.choice` samples from it. Build it once and token generation stops being magic.

In [ ]:
import numpy as np

# === Categorical Distribution = Token Prediction ===
vocab = ['the', 'cat', 'sat', 'on', 'mat', 'dog']
logits = np.array([2.1, 3.5, 0.8, 1.2, -0.3, 0.5])

# Softmax → PMF (probabilities sum to 1.0)
pmf = np.exp(logits) / np.exp(logits).sum()
print("Token PMF (softmax output):")
for t, p in sorted(zip(vocab, pmf), key=lambda x: -x[1]):
    bar = '█' * int(p * 40)
    print(f"  {t:6s} {p:.4f} {bar}")
print(f"  Sum: {pmf.sum():.6f}")  # Always 1.0

# Sample a token (this IS how LLMs generate text)
sampled = np.random.choice(vocab, p=pmf)
print(f"\nSampled token: '{sampled}'")


**Explanation**

This cell is a from-scratch demonstration of the categorical distribution over a tiny vocabulary. It runs softmax by hand to prove the outputs form a valid PMF, prints a bar chart of the probabilities, then draws one sample - the exact mechanic behind LLM text generation.

**How the code works - step by step**
- **`vocab` / `logits`** - a six-word vocabulary paired with raw (unnormalized) model scores.
- **`pmf = np.exp(logits) / np.exp(logits).sum()`** - softmax: exponentiate every logit and divide by the total, forcing all values non-negative and summing to 1.
- **the sorted print loop** - orders tokens by descending probability and draws a `█` bar scaled by `p * 40` so you can eyeball the distribution.
- **`pmf.sum()`** - printed as a sanity check; it is always ~1.0.
- **`np.random.choice(vocab, p=pmf)`** - samples one token weighted by the PMF - this IS how a model emits a token.

**In one line:** logits -> softmax -> a valid PMF -> sample a token.

**What you'll see:** A ranked table of the six tokens with their softmax probabilities and bar charts ('cat' and 'the' dominate), a `Sum: 1.000000` line, and one randomly sampled token such as `Sampled token: 'cat'`.

## 2 - Continuous variables and why the mean lies about latency

For a continuous random variable P(X = an exact value) is 0 - you reason about ranges and percentiles instead. API latency is the production case that matters: it is log-normal (always positive, right-skewed, long-tailed), so the mean looks healthy while the tail quietly hurts 1% of your users.

In [ ]:
import numpy as np

# === Log-Normal: THE distribution for API latency ===
np.random.seed(42)
# Simulate 10,000 LLM API calls
latencies = np.random.lognormal(mean=5.5, sigma=0.6, size=10000)

print("LLM API Latency Distribution:")
print(f"  mean:   {latencies.mean():.0f}ms")
print(f"  median: {np.median(latencies):.0f}ms")
print(f"  p50:    {np.percentile(latencies, 50):.0f}ms")
print(f"  p95:    {np.percentile(latencies, 95):.0f}ms")
print(f"  p99:    {np.percentile(latencies, 99):.0f}ms")
print(f"\n  mean vs median gap: {latencies.mean() - np.median(latencies):.0f}ms")
print("  → Right-skewed! mean is pulled by tail.")
print("  → ALWAYS use percentiles for latency SLOs, not mean.")


**Explanation**

A simulation-and-measurement cell, not a model call. It draws 10,000 fake latencies from a log-normal distribution and reports the summary statistics an SRE actually watches, making the gap between the mean and the tail percentiles concrete.

**How the code works - step by step**
- **`np.random.seed(42)`** - fixes the RNG so the numbers are reproducible run to run.
- **`np.random.lognormal(mean=5.5, sigma=0.6, size=10000)`** - simulates 10,000 API-call latencies from a right-skewed log-normal distribution.
- **`latencies.mean()` / `np.median(...)`** - the average vs the midpoint; the average sits higher because the tail drags it up.
- **`np.percentile(latencies, 50/95/99)`** - the p50/p95/p99 SLO numbers; p99 is far above the mean.
- **the `mean vs median gap` print** - quantifies the right-skew, motivating the closing rule to alert on percentiles, never the mean.

**In one line:** simulate log-normal latency, then let the percentiles expose the tail the mean hides.

**What you'll see:** A latency report printing mean, median, p50, p95, and p99 in milliseconds (mean noticeably above median, p99 far higher still), the mean-minus-median gap, and two lines noting the distribution is right-skewed so SLOs should use percentiles.

## 3 - The four distributions you actually reach for

Most GenAI probability work comes down to four distributions: Bernoulli (a binary event), Poisson (counts per interval), Gaussian (continuous noise / embeddings), and Beta (a posterior over a rate). This cell fires all four in a few lines so you can see which real problem each one owns.

In [ ]:
import numpy as np
from scipy import stats

# === Every distribution you need for GenAI ===
np.random.seed(42)

# 1. Bernoulli: user clicked? (binary)
clicks = np.random.binomial(1, p=0.03, size=10000)
print(f"CTR: {clicks.mean():.3f} (expected: 0.030)")

# 2. Poisson: API errors per hour
errors = np.random.poisson(lam=2.5, size=1000)
print(f"Errors/hr: mean={errors.mean():.1f}, P(0 errors)={np.mean(errors==0):.3f}")

# 3. Gaussian: embedding dimensions
embedding = np.random.randn(768)
print(f"Embedding: mean={embedding.mean():.3f}, std={embedding.std():.3f}")

# 4. Beta: A/B test posterior
# After 100 impressions, 8 clicks for variant A, 12 for B
posterior_a = stats.beta(8+1, 92+1)
posterior_b = stats.beta(12+1, 88+1)
p_b_better = np.mean(posterior_b.rvs(10000) > posterior_a.rvs(10000))
print(f"P(B > A): {p_b_better:.3f}")


**Explanation**

A four-in-one worked example that samples from each distribution against a realistic scenario. The Beta section is the payoff: it does a tiny Bayesian A/B test and returns an actual probability that variant B beats A, not a p-value.

**How the code works - step by step**
- **Bernoulli - `np.random.binomial(1, p=0.03, size=10000)`** - simulates 10,000 click/no-click trials at a 3% rate; `.mean()` recovers the CTR.
- **Poisson - `np.random.poisson(lam=2.5, size=1000)`** - models errors-per-hour at rate 2.5; reports the observed mean and the empirical P(0 errors).
- **Gaussian - `np.random.randn(768)`** - a 768-dim standard-normal vector standing in for an embedding; prints its mean (~0) and std (~1).
- **Beta - `stats.beta(8+1, 92+1)` and `stats.beta(12+1, 88+1)`** - posteriors over A's and B's click rates (successes+1, failures+1); drawing 10,000 samples from each and comparing gives `P(B > A)`.

**In one line:** Bernoulli for clicks, Poisson for counts, Gaussian for embeddings, Beta for 'which variant wins.'

**What you'll see:** Four lines: an observed CTR near 0.030, an errors/hour mean near 2.5 with P(0 errors) around 0.08, an embedding mean near 0 with std near 1, and a `P(B > A)` probability (roughly 0.7-0.8) for the Beta A/B comparison.

## 4 - Where signal processing meets statistics: the spectrogram

The Wiener-Khinchin bridge links a signal's frequency content to its statistics. This cell hand-builds the front half of a speech pipeline: take a noisy audio signal and run a Short-Time Fourier Transform to produce the time-by-frequency spectrogram that a model like Whisper consumes.

In [ ]:
import numpy as np

# === Whisper pipeline = signal processing + probability ===
# Simulate a simple audio signal (2 frequencies)
sr = 16000  # 16kHz sample rate (Whisper's input)
t = np.arange(0, 1.0, 1/sr)  # 1 second
signal = np.sin(2 * np.pi * 440 * t) + 0.5 * np.sin(2 * np.pi * 880 * t)
signal += np.random.randn(len(t)) * 0.1  # noise

# STFT: Short-Time Fourier Transform (windowed FFT)
window_size = 400  # 25ms window
hop = 160  # 10ms hop
n_frames = (len(signal) - window_size) // hop + 1

spectrogram = []
for i in range(n_frames):
    frame = signal[i*hop : i*hop + window_size]
    windowed = frame * np.hanning(window_size)
    fft_mag = np.abs(np.fft.rfft(windowed))[:80]  # first 80 bins
    spectrogram.append(fft_mag)

spectrogram = np.array(spectrogram)
print(f"Spectrogram shape: {spectrogram.shape}")
print(f"  = {n_frames} time frames × 80 frequency bins")
print(f"  This feeds into Whisper's encoder!")


**Explanation**

A from-scratch STFT implemented with a plain Python loop over windowed frames. It fabricates a two-tone noisy signal, slides a Hann-windowed FFT across it, and stacks the magnitude spectra into a 2D spectrogram - showing exactly what shape of tensor an audio encoder ingests.

**How the code works - step by step**
- **signal construction** - at a 16kHz sample rate, sums 440Hz and 880Hz sine waves over one second and adds Gaussian noise.
- **`window_size = 400` / `hop = 160`** - a 25ms analysis window advancing in 10ms steps; `n_frames` is how many windows fit.
- **the frame loop** - for each frame: slice it, multiply by `np.hanning` to taper the edges, take `np.fft.rfft`, keep the magnitude of the first 80 frequency bins.
- **`spectrogram = np.array(spectrogram)`** - stacks the per-frame spectra into a `(n_frames, 80)` matrix.

**In one line:** slide a windowed FFT across audio to build the time x frequency image a speech encoder reads.

**What you'll see:** Three printed lines: the spectrogram shape (`(98, 80)`), a note that this is 98 time frames x 80 frequency bins, and a line saying this feeds into Whisper's encoder.

Four cells, four distributions-in-context: a categorical PMF is literally how a model picks its next token, log-normal latency is why you alert on p99 not the mean, and Bernoulli/Poisson/Gaussian/Beta cover clicks, error rates, embeddings, and Bayesian A/B tests. The final cell shows probability and signal processing meeting in a spectrogram - the same tensor a Whisper encoder ingests. Carry these forward: the Beta-posterior A/B pattern returns in Module 12, and the spectrogram pipeline in Module 10 (multimodal).